# Lab 03: Introduction to Machine Learning
**Week 4 — October 5**

A fictional housing analyst wants to predict sale prices from home size. In this lab, you will identify the learning task, measure prediction error, take one gradient descent step, and look for overfitting. The data are synthetic and do not describe real homes.

By the end, you should be able to:
- Distinguish supervised regression, supervised classification, and unsupervised learning.
- Separate a feature from its target and use held-out validation data.
- Calculate mean absolute error (MAE) and mean squared error (MSE).
- Use a gradient to improve a simple prediction line.
- Explain why lower training error does not guarantee better predictions on new data.

Work from top to bottom. Run cells with **Shift + Enter**. Replace every **...** in an answer cell and complete the written responses. You can work with a neighbor and ask a tutor/TA for help.

## Setup

Keep **house_prices.csv** beside this notebook. Each row is one fictional house.

| Column | Meaning |
|---|---|
| size_k_sqft | Home size in thousands of square feet |
| price_100k | Sale price in hundreds of thousands of dollars |
| split | Whether this row is for training or validation |

We fit on **training** rows. Validation rows help us see how a model behaves on examples it did not use for fitting. A separate final test set would be reserved for the very end of a real project.

The setup standardizes size using the **training** mean and standard deviation, then applies those same values to validation rows. This makes the gradient descent step easier to tune without learning anything from validation prices.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import otter

grader = otter.Notebook("Lab03.ipynb", tests_dir="tests/lab03")
homes = pd.read_csv("house_prices.csv")
train = homes.loc[homes["split"] == "train"].copy()
validation = homes.loc[homes["split"] == "validation"].copy()

size_mean = train["size_k_sqft"].mean()
size_std = train["size_k_sqft"].std(ddof=0)
train["size_scaled"] = (train["size_k_sqft"] - size_mean) / size_std
validation["size_scaled"] = (validation["size_k_sqft"] - size_mean) / size_std

print("Training rows:", len(train), "| Validation rows:", len(validation))
homes.head()

### Q1. What kind of learning is this?

Predicting **price_100k** from **size_k_sqft** is supervised **regression**. Explain why it is supervised and why it is regression. Then classify these two tasks:

1. Predict whether a home sells **above asking price** from past sales labeled yes/no.
2. Group homes by size and neighborhood **without** a price or category to predict.

> Write your answer here.

## Part 1: Features, predictions, and loss

A **feature** is an input available when we make the prediction. A **target** is the value we want to predict. We will use standardized size as our only feature.

### Q2. Select the training feature and target

Save a one-column **DataFrame** containing **size_scaled** as **X_train**. Save the **price_100k** Series as **y_train**. The target should not be in the feature table.

**Hint:** Use double brackets for a one-column DataFrame.

In [ ]:
X_train = train[[...]]
y_train = train[...]
X_train.head()

In [ ]:
grader.check("q2")

### Q3. Make a simple baseline prediction

Before fitting a line, try a constant model: predict the **mean training price** for every training house. Save its predictions as a NumPy array named **baseline_train_pred**.

**Hint:** Use **np.full(number_of_rows, value)**. The number of rows is **len(y_train)**.

In [ ]:
baseline_train_pred = np.full(..., ...)
baseline_train_pred[:5]

In [ ]:
grader.check("q3")

### Q4. Measure the baseline's error

For each house, the error is **actual price − predicted price**. The **MAE** averages absolute errors; the **MSE** averages squared errors:

$$\mathrm{MAE}=\frac{1}{n}\sum_i |y_i-\hat y_i| \qquad
\mathrm{MSE}=\frac{1}{n}\sum_i (y_i-\hat y_i)^2.$$

Compute both for the training rows. Save them as **baseline_mae** and **baseline_mse**. Use **np.mean**, **np.abs**, and **y_train.to_numpy()**.

**Units:** MAE is in hundreds of thousands of dollars. MSE is in squared units, so its number is not directly a dollar amount.

In [ ]:
baseline_mae = ...
baseline_mse = ...
print("Baseline MAE:", baseline_mae)
print("Baseline MSE:", baseline_mse)

In [ ]:
grader.check("q4")

**Your response:** Which loss would react more strongly to one very large prediction error? Explain using the formulas.

> Write your answer here.

## Part 2: Take one gradient descent step

Our line predicts **price = w × standardized size + b**. The slope **w** and intercept **b** are parameters the model learns. We start with **w = 0**, so the line makes the same baseline prediction for every house. The **learning rate** controls the step size.

For MSE, calculate the two gradients using the current predictions:

$$g_w=-2\,\mathrm{mean}\!\left(x(y-\hat y)\right), \qquad
g_b=-2\,\mathrm{mean}(y-\hat y).$$

Then update each parameter in the direction **opposite** its gradient.

In [ ]:
x = X_train["size_scaled"].to_numpy()
y = y_train.to_numpy()
w = 0.0
b = float(y_train.mean())
learning_rate = 0.1
print("Starting slope:", w, "| Starting intercept:", b)

### Q5. Calculate the gradients and update the line

Fill in the five blanks. **current_pred** should use the starting **w** and **b**. Then compute **grad_w**, **grad_b**, **new_w**, and **new_b**.

**Hint:** The update rule is **new parameter = old parameter − learning_rate × gradient**.

In [ ]:
current_pred = ...
grad_w = ...
grad_b = ...
new_w = ...
new_b = ...
print("Gradient:", grad_w, grad_b)
print("Updated line:", new_w, new_b)

In [ ]:
grader.check("q5")

### Q6. Check the training and validation errors

Use the updated line to predict both training and validation prices. Save the prediction arrays as **updated_train_pred** and **validation_pred**. Compute each MSE as **updated_train_mse** and **validation_mse**.

Use **validation["size_scaled"].to_numpy()** for validation inputs. Do not calculate a new mean or standard deviation from validation rows.

In [ ]:
updated_train_pred = ...
updated_train_mse = ...
validation_pred = ...
validation_mse = ...
print("Baseline training MSE:", round(baseline_mse, 4))
print("Updated training MSE:", round(updated_train_mse, 4))
print("Validation MSE:", round(validation_mse, 4))

In [ ]:
grader.check("q6")

Run the plot below after Q6. Blue points were used to fit the line; orange points were held out for validation.

In [ ]:
grid = np.linspace(train["size_scaled"].min(), train["size_scaled"].max(), 100)
plt.scatter(train["size_scaled"], y_train, label="Training houses")
plt.scatter(validation["size_scaled"], validation["price_100k"], label="Validation houses")
plt.plot(grid, w * grid + b, "--", label="Before the step")
plt.plot(grid, new_w * grid + new_b, label="After the step")
plt.xlabel("Standardized home size")
plt.ylabel("Price ($100k)")
plt.title("One gradient descent step improves the prediction line")
plt.legend()
plt.show()

**Your response:** Did the training MSE fall after one step? If the learning rate were much too large, what could happen to the loss?

> Write your answer here.

## Part 3: Generalization and overfitting

A flexible model can fit the training data closely yet perform worse on new examples. The next cell fits two polynomial curves using only the training rows. Run it and compare their training and validation MSE. This is a demonstration; you do not need to write polynomial-fitting code.

In [ ]:
validation_x = validation["size_scaled"].to_numpy()
validation_y = validation["price_100k"].to_numpy()
plot_x = np.linspace(x.min(), x.max(), 200)

plt.figure(figsize=(8, 5))
plt.scatter(x, y, label="Training", color="tab:blue")
plt.scatter(validation_x, validation_y, label="Validation", color="tab:orange")
for degree, color in [(1, "tab:green"), (10, "tab:red")]:
    coefficients = np.polyfit(x, y, degree)
    train_predictions = np.polyval(coefficients, x)
    held_out_predictions = np.polyval(coefficients, validation_x)
    train_error = np.mean((y - train_predictions) ** 2)
    held_out_error = np.mean((validation_y - held_out_predictions) ** 2)
    print(f"Degree {degree}: train MSE = {train_error:.4f}, validation MSE = {held_out_error:.4f}")
    plt.plot(plot_x, np.polyval(coefficients, plot_x), color=color, label=f"Degree {degree}")

plt.xlabel("Standardized home size")
plt.ylabel("Price ($100k)")
plt.title("A lower training error can hide overfitting")
plt.ylim(1.5, 4.6)
plt.legend()
plt.show()

### Q7. Interpret the comparison

Which degree has the **lower training MSE**? Which has the **lower validation MSE**? Identify the model that appears to overfit and explain what the gap between its two errors tells you.

Remember: we can compare candidate models using validation data, but in a real project we would keep a separate test set untouched until after choosing the model.

> Write your answer here.

## Wrap-up

In 2–3 sentences, explain how **loss**, **gradient descent**, and **validation error** each help with learning a model that generalizes.

> Write your answer here.

### Check your code

Run the next cell to recheck all five coding exercises. Otter checks code results; it does not assess Q1, Q7, or other written responses. If you change an earlier answer, rerun the later cells before checking again.

**Before submitting:** Run the completed notebook from top to bottom, complete every written response, and save your **Lab03.ipynb** for the lab assignment on bCourses. Keep **house_prices.csv** and the **tests** folder beside the notebook while working.

In [ ]:
grader.check_all()